# Demo 00 - Reviewing a Database

In this first demo, we are going to review what we can learn about this data set from SQL Server.

First, let's set up our database connection.

In [ ]:
import tomllib
from sqlalchemy import create_engine

with open("../config.toml", "rb") as f:
    config = tomllib.load(f)

server = config["database"]["server"]
database = config["database"]["name"]
username = config["database"]["username"]
password = config["database"]["password"]

conn_str = f"mssql+pyodbc://{username}:{password}@{server}/{database}?driver=ODBC+Driver+18+for+SQL+Server&TrustServerCertificate=yes"
engine = create_engine(conn_str)

%load_ext sql
%sql engine

Here is one of my favorite queries; it estimates the number of rows from statistics. This number might be off by a little bit if statistics are old or if you are constantly updating the data, but it's usually really close; furthermore, at this point, we just want to see a rough idea of what our largest tables are, so if numbers are off by a few, that's not a big concern.

In [ ]:
%%sql
SELECT
    CONCAT(OBJECT_SCHEMA_NAME(s.object_id), '.', OBJECT_NAME(s.object_id)) AS TableName,
    SUM(row_count) AS NumberOfRows
FROM sys.dm_db_partition_stats s
    INNER JOIN sys.tables t
        ON s.object_id = t.object_id
WHERE
    s.index_id < 2
GROUP BY
    OBJECT_SCHEMA_NAME(s.object_id),
    OBJECT_NAME(s.object_id)
ORDER BY
    NumberOfRows DESC;

We learned from this that `dbo.LineItem` is the largest table by an order of magnitude, so it's going to be one of the most important. Given its size (and the fact that it's not named something like "Log" or "Backup" or "DeleteMe"), that makes it our first target. Let's look at data types, constraints, and keys.

## Columns and Data Types

In [ ]:
%%sql
SELECT
    c.column_id,
    c.name AS ColumnName,
    ty.name AS ColumnType,
    c.max_length,
    c.precision,
    c.scale,
    c.is_nullable,
    c.is_identity,
    c.is_computed
FROM sys.tables t
    INNER JOIN sys.columns c
        ON t.object_id = c.object_id
    INNER JOIN sys.types ty
        ON c.system_type_id = ty.system_type_id
        AND c.user_type_id = ty.user_type_id
WHERE
    t.name = N'LineItem'
    AND t.schema_id = SCHEMA_ID('dbo');

### Sample Data

After seeing what the data types look like, let's take a brief look at some sample data, just to get a feel for the table. This will help us with things like how often we see null columns and a rough idea of data distribution.

In [ ]:
%%sql
SELECT TOP(100) *
FROM dbo.LineItem;

## Non-Key Constraints

There are two major types of key constraint we want to look at: check constraints and default constraints.

### Check Constraints

In [ ]:
%%sql
SELECT
    cc.name AS ConstraintName,
    cc.definition
FROM sys.check_constraints cc
    INNER JOIN sys.tables t
        ON cc.parent_object_id = t.object_id
WHERE
    t.name = N'LineItem'
    AND t.schema_id = SCHEMA_ID('dbo')

### Default Constraints

In this case, we don't have any default constraints, so no need to go further.

In [ ]:
%%sql
SELECT *
FROM sys.default_constraints dc;

## Key Constraints

There are three major forms of key constraint we care about. We want to know what the primary key is, we want to know if there are any unique keys (or unique indexes), and we want to know about foreign keys.

### Primary and Unique Keys

Note that this query uses the `STRING_AGG()` function, which was introduced in SQL Server 2017. It also works in Azure SQL Database.

In [ ]:
%%sql
SELECT
    kc.name AS PrimaryKeyName,
    kc.type AS ConstraintType,
    kc.type_desc AS ConstraintTypeDesc,
    STRING_AGG(c.name, ',') AS ConstraintColumns
FROM sys.key_constraints kc
    INNER JOIN sys.indexes i
        ON kc.parent_object_id = i.object_id
    INNER JOIN sys.index_columns ic
        ON ic.index_id = i.index_id
        AND ic.object_id = i.object_id
    INNER JOIN sys.columns c
        ON c.object_id = ic.object_id
        AND c.column_id = ic.column_id
    INNER JOIN sys.tables t
        ON kc.parent_object_id = t.object_id
WHERE
    t.name = N'LineItem'
    AND t.schema_id = SCHEMA_ID('dbo')
GROUP BY
    kc.name,
    kc.type,
    kc.type_desc

### Foreign Key Constraints

The last thing we will check in this review is foreign keys, so we can see how tables connect together.

In [ ]:
%%sql
SELECT
    fk.name AS ForeignKeyName,
    tRef.name AS ReferencedTableName,
    STRING_AGG(c.name, ',') AS BaseColumnName,
    STRING_AGG(cRef.name, ',') AS ReferencedColumnName
FROM sys.foreign_keys fk
    INNER JOIN sys.tables t
        ON fk.parent_object_id = t.object_id
    INNER JOIN sys.foreign_key_columns fkc
        ON fk.object_id = fkc.constraint_object_id
    INNER JOIN sys.columns c
        ON fkc.parent_column_id = c.column_id
        AND fkc.parent_object_id = c.object_id
    INNER JOIN sys.columns cRef
        ON fkc.referenced_column_id = cRef.column_id
        AND fkc.referenced_object_id = cRef.object_id
    INNER JOIN sys.tables tRef
        ON fk.referenced_object_id = tRef.object_id
WHERE
    t.name = N'LineItem'
    AND t.schema_id = SCHEMA_ID('dbo')
GROUP BY
    fk.name,
    tRef.name

## Analyze Recursively

After reviewing one table and getting a feeling for it, we want to move on to the next table in our foreign key list, which will be `dbo.Bus`. We'll repeat the process for it, then on and on. Mark off tables you've already analyzed to prevent accidentally repeating work.

## Rinse and Repeat

If you get to the point where you have recursively analyzed all of the tables in a cluster, go back to the table list and start again from the largest remaining table.